# ACCESS-AIS3 - Development workflow

This workbook presents v0.1 of the ACCESS-AIS3 Antarctic model configuration. Development is completed in this workbook and will be converted/exported to an executable script once complete.

In [ ]:
import pyissm
import ccdtools
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import os

TODO:
- Review BCs - No Dirichlet at this point
- Use relative file paths
- Use MIPKIT rather than individual data files.

In [ ]:
## ------------------------------------
## Configure options
## ------------------------------------

# Should plots be generated?
plot = False
diagnostics = True
save = True
inversion_sensitivity = False

# Define execution directory
execution_dir = '/g/data/au88/lb9857/access-ais3/models'

# Define domain_file
domain_file = ('/home/565/lb9857/gitRepos/ACCESS-AIS3/assets/ais_domain.exp')

# Define param_file
param_file = ('/home/565/lb9857/gitRepos/ACCESS-AIS3/config/ais_0.1_param.py')

# Define cluster requirements
cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR']
cluster.executionpath = execution_dir
cluster.storage = 'gdata/au88+gdata/vk83'
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm/2025.11.0']
cluster.np = 32
cluster.memory = 100
cluster.time = 60*24
cluster.login = 'lb9857'
cluster.project = 'au88'

# List all steps for clarity
all_steps = [
    'process_domain',
    'mesh',
    'param',
    'extrude',
    'ho_rheology_floating'
]

# Define steps to run
# steps = ['process_domain']
# steps = ['mesh', 'param', 'extrude']
steps = ['ho_rheology_floating_inv']

In [ ]:
## ------------------------------------
## Initialise Data Catalog
## ------------------------------------
catalog = ccdtools.catalog.DataCatalog()
bedmachine_data = catalog.load_dataset('measures_bedmachine_antarctica', version = 'v3')
velocity_data = catalog.load_dataset('measures_insar_based_antarctica_ice_velocity_map', version = 'v2')
measures_coastline = catalog.load_dataset('measures_antarctic_boundaries', subdataset = 'coastline')

In [ ]:
## ------------------------------------
## Process domain file
## ------------------------------------

if 'process_domain' in steps:

    print("-------------------------------------------------------------")
    print(f" PROCESSING DOMAIN FILE"                                     )
    print("-------------------------------------------------------------")

    # Buffer coastline polygon by 100 km
    print(f" - Buffering coastline...")
    coastline_100km_buffer = measures_coastline.buffer(100000)

    # Write buffered extent to file for use as model domain
    print(f" - Saving to file...")
    pyissm.tools.exp.gdf_to_exp(coastline_100km_buffer, '/home/565/lb9857/gitRepos/ACCESS-AIS3/assets/ais_domain.exp')


In [ ]:
## ------------------------------------
## Create mesh
## ------------------------------------
if 'mesh' in steps:

    print("-------------------------------------------------------------")
    print(f" GENERATING MESH"                                            )
    print("-------------------------------------------------------------")

    # Create empty model with initial 10e3 resolution mesh
    md = pyissm.model.mesh.triangle(pyissm.model.Model(), domain_file, 10e3)
    
    # Remesh the model twice to refine based on velocity and bedmachine mask
    for i in range(2):

        print(f"REFINEMENT ITERATION: {i+1}")

        # Interpolate velocities onto mesh
        print(f"\n-- Interpolating MEaSURES v2 Velocities...")
        vx = pyissm.data.interp.xr_to_mesh(velocity_data, 'VX', md.mesh.x, md.mesh.y)
        vy = pyissm.data.interp.xr_to_mesh(velocity_data, 'VY', md.mesh.x, md.mesh.y)
        vel = np.sqrt(vx**2 + vy**2)
        
        if diagnostics:
            print(f"\nVELOCITY DIAGNOSTICS:")
            print(f"    Max velocity: {np.nanmax(vel):.2f} m/yr")
            print(f"    Min velocity: {np.nanmin(vel):.2f} m/yr")
    
        if plot:
            pyissm.plot.plot_model_field(md, vel, cmap = 'PuOr',show_cbar = True, cbar_kwargs = {'label': 'Velocity (m/a)'}); plt.show(block = False)

        # Interpolate ice mask onto mesh
        print(f"\n-- Interpolating Bedmachine v3 Ice Mask...")
        mask = pyissm.data.interp.xr_to_mesh(bedmachine_data, 'mask', md.mesh.x, md.mesh.y, interpolation_type = 'nearest')
        
        if diagnostics:
            unique_vals, counts = np.unique(mask, return_counts=True)
            print(f"\nMASK DIAGNOSTICS:")
            for val, count in zip(unique_vals, counts):
                print(f"    Value {val}: {count} occurrences")

        # Fill nan values and set to 0 in ocean and ice-free areas
        print(f"\n-- Set Velocity to 0 where NaNs exist or mask < 2...")
        vel[np.isnan(vel) | (mask < 2)] = 0.0
        
        if plot:
            pyissm.plot.plot_model_field(md, mask, show_cbar = True, cbar_kwargs = {'label': 'Ice Mask'}); plt.show(block = False)

        # Define min/max vertex lengths in specific regions
        print(f"\n-- Setting min/max vertex dimensions...")
        hmax_v = np.full(md.mesh.numberofvertices, np.nan)
        hmin_v = np.full(md.mesh.numberofvertices, np.nan)
    
        hmax_v[(vel > 50) & (mask == 2)] = 1500 # Max length on fast-flowing grounded ice
        hmin_v[(mask == 3)] = 500 # Min length on ice shelves
        hmax_v[(mask == 3)] = 5000 # Max length on ice shelves

        # Adjust mesh with specified metrics
        print(f"\n-- Remeshing with specified metrics...")
        md = pyissm.model.mesh.bamg(md,
                                    hmin = 50,
                                    hmax = 50e3,
                                    hmaxVertices = hmax_v,
                                    hminVertices = hmin_v,
                                    maxnbv = 2e6,
                                    field = vel,
                                    err = 1,
                                    gradation = 1.2)
        
        # Remove bamg private data to allow additional remeshes
        md.private.bamg = {}
    
        if diagnostics:
            print(f"\nMESH DIAGNOSTICS:")
            print(f"   Number of elements: {md.mesh.numberofelements}")
            print(f"   Number of vertices: {md.mesh.numberofvertices}")
    
    # Set georefernce information
    [md.mesh.lat, md.mesh.long] = pyissm.tools.general.xy_to_ll(md.mesh.x, md.mesh.y, -1)
    md.mesh.epsg = 3031
    
    print(f"\nFinal mesh: {md.mesh.numberofvertices} nodes; {md.mesh.numberofelements} elements")
    
    if plot:
        areas = pyissm.model.mesh.get_element_areas_volumes(md.mesh.elements, md.mesh.x, md.mesh.y)
        # A = l^2 * sqrt(3) / 4 is area for equilateral triangle
        # np.sqrt(A*2) / 1e3 is the rough approximation of element edge length in km
        fig, ax = pyissm.plot.plot_model_field(md,
                                               np.sqrt(areas*2)/1e3,
                                               show_cbar = True,
                                               vmin = 0.25,
                                               vmax = 10,
                                               plot_data_on='elements',
                                               cmap = 'plasma_r',
                                               cbar_kwargs = {'label': 'Approx. element edge length (km)'})
        ax.set_title('Final Mesh: Velocity-adapted w/ 2 refinement passes')
        plt.show(block = False)

    if save:
        print(f"\nSaving model to {execution_dir}/AIS3_mesh.nc")
        pyissm.model.io.save_model(md, f'{execution_dir}/AIS3_mesh.nc')

In [ ]:
## ------------------------------------
## Parameterise model
## ------------------------------------
if 'param' in steps:

    print("-------------------------------------------------------------")
    print(f" PARAMETERIZING MODEL"                                       )
    print("-------------------------------------------------------------")

    print(f"-- Loading model mesh...")
    md = pyissm.model.io.load_model(f'{execution_dir}/AIS3_mesh.nc')

    print(f"-- Parameterising model using {param_file}...")
    md = pyissm.model.param.parameterize(md, param_file)

    print(f"-- Setting Boundary Conditions...")
    # -------- Set Stress Balance BCs --------
    ## Initialize empty fields
    md.stressbalance.spcvx = np.nan * np.ones(md.mesh.numberofvertices)
    md.stressbalance.spcvy = np.nan * np.ones(md.mesh.numberofvertices)
    md.stressbalance.spcvz = np.nan * np.ones(md.mesh.numberofvertices)
    
    ## Find ice nodes on the edge of the domain 
    pos = (md.mask.ice_levelset < 0) & (md.mesh.vertexonboundary.astype(bool))
    
    ## Set Dirichlet BCs on VX and VY fields based on initial velocities; Set VZ as 0
    md.stressbalance.spcvx[pos] = md.initialization.vx[pos]
    md.stressbalance.spcvy[pos] = md.initialization.vy[pos]
    md.stressbalance.spcvz[pos] = 0 #TODO: Specify this here?

    md.stressbalance.referential = np.nan * np.ones((md.mesh.numberofvertices, 6))
    md.stressbalance.loadingforce = np.zeros((md.mesh.numberofvertices, 3))

    # -------- Set thermal Balance BCs --------
    md.thermal.spctemperature = md.initialization.temperature.copy()
  

    if diagnostics:
        print(f"\nMASK DIAGNOSTICS:")
        print(f" - Ice Levelset:")        
        unique_vals, counts = np.unique(md.mask.ice_levelset, return_counts=True)
        for val, count in zip(unique_vals, counts):
            print(f"    Value {val}: {count} occurrences")

        print(f" Ocean Levelset (Binary):")
        ocean_binary = md.mask.ocean_levelset <= 0
        unique_vals, counts = np.unique(ocean_binary, return_counts=True)
        for val, count in zip(unique_vals, counts):
            print(f"    Value {val}: {count} occurrences")

        print(f"\nGEOMETRY DIAGNOSTICS:")
        print(f" - Surface elevation:")
        print(f"   min = {np.min(md.geometry.surface):.2f} m")
        print(f"   max = {np.max(md.geometry.surface):.2f} m")
        print(f" - Bed elevation:")
        print(f"   min = {np.min(md.geometry.bed):.2f} m")
        print(f"   max = {np.max(md.geometry.bed):.2f} m")
        print(f" - Thickness:")
        print(f"   min = {np.min(md.geometry.thickness):.2f} m")
        print(f"   max = {np.max(md.geometry.thickness):.2f} m")

        print(f"VELOCITY DIAGNOSTICS:")
        print(f"   Min observed vx: {np.min(md.inversion.vx_obs):.2f} m/yr")
        print(f"   Max observed vx: {np.max(md.inversion.vx_obs):.2f} m/yr")
        print(f"   Min observed vy: {np.min(md.inversion.vy_obs):.2f} m/yr")
        print(f"   Max observed vy: {np.max(md.inversion.vy_obs):.2f} m/yr")
        print(f"   Min observed vel: {np.min(md.inversion.vel_obs):.2f} m/yr")
        print(f"   Max observed vel: {np.max(md.inversion.vel_obs):.2f} m/yr")

        print(f"INITIAL PRESSURE DIAGNOSTICS:")
        print(f"   Min initial pressure: {np.min(md.initialization.pressure):.2f} Pa")
        print(f"   Max initial pressure: {np.max(md.initialization.pressure):.2f} Pa")

        print(f"GEOTHERMAL HEAT FLOW DIAGNOSTICS:")
        print(f"   Min geothermal heat flux: {np.min(md.basalforcings.geothermalflux):.7f} mW/m2")
        print(f"   Max geothermal heat flux: {np.max(md.basalforcings.geothermalflux):.7f} mW/m2")

        print(f"INITIAL TEMPERATURE DIAGNOSTICS:")
        print(f"   Min initial temp: {np.min(md.initialization.temperature):.2f} K")
        print(f"   Max initial temp: {np.max(md.initialization.temperature):.2f} K")

    if save:
        print(f"\nSaving model to {execution_dir}/AIS3_param.nc")
        pyissm.model.io.save_model(md, f'{execution_dir}/AIS3_param.nc')

In [ ]:
## ------------------------------------
## Extrude model
## ------------------------------------
if 'extrude' in steps:

    print("-------------------------------------------------------------")
    print(f" EXTRUDING MODEL"                                       )
    print("-------------------------------------------------------------")

    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{execution_dir}/AIS3_param.nc')

    print(f"-- Extruding model to 3D...")
    md = md.extrude(10, 1.1)

    print(f"-- Set flow equation to HO...")
    md = pyissm.model.param.set_flow_equation(md, HO = 'all')

    if save:
        print(f"\nSaving model to {execution_dir}/AIS3_extrude.nc")
        pyissm.model.io.save_model(md, f'{execution_dir}/AIS3_extrude.nc')

In [ ]:
## ------------------------------------
## HO Rheology Inversion - Floating Ice
## ------------------------------------
if 'ho_rheology_floating_inv' in steps:
    
    print("-------------------------------------------------------------")
    print(f" HO RHEOLOGY INVERSION - FLOATING ICE"                       )
    print("-------------------------------------------------------------")
    
    print(f"-- Loading extruded model...")
    md = pyissm.model.io.load_model(f'{execution_dir}/AIS3_extrude.nc')
    
    print(f"-- Define general control parameters...")
    md.inversion.iscontrol = 1
    md.verbose.solution = 0
    md.verbose.qmu = 0
    md.verbose.control = 1
    
    # Run inversion_sensitivity process to select optimal inversion parameters
    if inversion_sensitivity:
        warning.warn(f"inversion_sensitivity functionality is not yet implemented. Set inversion_sensitivity = False to run inversion with best-guess.")
    else:    
        print(f"-- Defining inversion parameters...")
        md.inversion = pyissm.model.classes.inversion.m1qn3(md.inversion)
        md.inversion.iscontrol = 1
        md.inversion.control_parameters = ['MaterialsRheologyBbar']
        md.inversion.cost_functions = [101, 103, 502]
        md.inversion.cost_functions_coefficients = np.ones((md.mesh.numberofvertices, 3))
        md.inversion.cost_functions_coefficients[:, 0] = 2000
        md.inversion.cost_functions_coefficients[:, 1] = 40
        md.inversion.cost_functions_coefficients[:, 2] = 1e-16
        pos = md.inversion.vel_obs == 0
        md.inversion.cost_functions_coefficients[pos, 0:2] = 0 # Column index is exclusive, so 0:2 sets both 0 and 1 to zero
        md.inversion.min_parameters = pyissm.tools.materials.cuffey(273.15 - 0) * np.ones((md.mesh.numberofvertices, ))
        md.inversion.max_parameters = pyissm.tools.materials.cuffey(273.15 - 70) * np.ones((md.mesh.numberofvertices, ))
        md.inversion.maxsteps = 500
        md.inversion.maxiter = 200
    
        print(f"-- Extracting floating ice only...")
        mask = (md.mask.ocean_levelset < 0) & (md.mask.ice_levelset < 0)
        mds = md.extract(mask)
    
        print(f"-- Assigning cluster and updating settings...")
        mds.cluster = cluster
        mds.settings.waitonlock = 0
        mds.miscellaneous.name = 'AIS3_ho_rheology_floating_inv'
    
        # Solve inversion
        if save:
            print(f"\nSaving model to {execution_dir}/AIS3_ho_rheology_floating_inv.nc")
            mds = pyissm.model.execute.solve(mds, 'Stressbalance', load_only = True, runtime_name = False)
            pyissm.model.io.save_model(mds, f'{execution_dir}/AIS3_ho_rheology_floating_inv.nc')
    
        else:
            print(f"-- Running Rheology B inversion on floating ice...")    
            mds = pyissm.model.execute.solve(mds, 'Stressbalance', load_only = False, runtime_name = False)
